# 11.2 BPTT와 numpy RNN 언어모델 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter11_2_bptt_char_rnn.ipynb)

책 본문: [11.2 BPTT와 numpy RNN 언어모델 실습](https://smhanlab.com/book-ml/kor/ml1/chapter11/2.html)

이 노트북은 11.2절의 내용을 코드로 끝까지 실행합니다:
(1) RNN의 **순전파**와 **BPTT**(시간을 펼쳐서 역전파)를 numpy로 구현,
(2) 시점 2개짜리 RNN에 BPTT를 **수치 미분과 교차 검증**(9.1절의 기법),
(3) "abc" 반복 문자열을 예측하는 **문자 단위 RNN 언어모델**을 500 에폭 학습 → "abcabcabcabc" 재현 + 손실 곡선,
(4) **그래디언트 클리핑**이 없는 경우 가중치가 발산하는 모습을 정량 확인,
(5) **잘라내기 BPTT**(truncated BPTT)가 전체 BPTT와 거의 같은 그래디언트를 주는 이유 확인.

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)
import os
os.makedirs(IMG, exist_ok=True)
np.random.seed(42)

## 1. RNN 순전파와 BPTT (numpy)

본문 "BPTT 유도" 섹션의 코드를 그대로 둔다. 핵심은 두 줄:
- `dh = Why.T @ dp + dh_next` — 본 시점 출력 오차 + **더 새로운 시점에서 흘러온** 그래디언트의 합.
- `dWhh += ...`, `dWxh += ...` — 가중치가 모든 시점에서 **공유**되므로 각 시점 기여를 **더한다**(`+=`).
  (이 `+=`를 `=`로 쓰면 BPTT가 틀어진다 — 본문 "자주 하는 실수" 참고.)

In [2]:
def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def one_hot(i, V):
    v = np.zeros(V); v[i] = 1.0; return v

def forward(inputs, Wxh, Whh, Why, bh, by, h0=None):
    """순전파: hs[t] = h_t, ps[t] = p_t 반환"""
    H = Wxh.shape[0]; V = Wxh.shape[1]
    h = np.zeros(H) if h0 is None else h0
    hs, ps, hps = [], [], []
    for idx in inputs:
        x = one_hot(idx, V)
        hps.append(h)                                   # h_{t-1} (이전 은닉 상태)
        h = np.tanh(Wxh @ x + Whh @ h + bh)             # h_t
        hs.append(h)
        ps.append(softmax(Why @ h + by))                # p_t
    return hs, ps, hps

def bptt(inputs, targets, Wxh, Whh, Why, bh, by, clip=None):
    """BPTT: 시간 T-1 → 0 순으로 역전파. 손실과 5개 파라미터의 그래디언트 반환."""
    H = Wxh.shape[0]; V = Wxh.shape[1]
    hs, ps, hps = forward(inputs, Wxh, Whh, Why, bh, by)
    dWxh = np.zeros_like(Wxh); dWhh = np.zeros_like(Whh)
    dWhy = np.zeros_like(Why); dbh = np.zeros(H); dby = np.zeros(V)
    dh_next = np.zeros(H)          # ∂L/∂h_{t+1} (최신 시점에서 흘러옴)
    loss = 0.0
    for t in reversed(range(len(inputs))):
        p = ps[t]
        dp = p.copy(); dp[targets[t]] -= 1.0            # ∂L/∂z_t^{out} = p_t - onehot
        loss += -np.log(p[targets[t]])
        dWhy += np.outer(dp, hs[t]); dby += dp          # (A) 본 시점 출력
        dh = Why.T @ dp + dh_next                       # ∂L/∂h_t = (A)+(B)
        dz = dh * (1 - hs[t]**2)                        # ∂L/∂z_t = dh ⊙ tanh'(z_t)
        dh_next = Whh.T @ dz                            # ∂L/∂h_{t-1} (다음 루프에서 B로)
        dWhh += np.outer(dz, hps[t]); dbh += dz
        dWxh += np.outer(dz, one_hot(inputs[t], V))
    if clip is not None:
        for d in (dWxh, dWhh, dWhy, dbh, dby):
            np.clip(d, -clip, clip, out=d)
    return loss, (dWxh, dWhh, dWhy, dbh, dby)

## 2. 손계산 예제: 시점 2개짜리 RNN에 BPTT + 수치 미분 검증

본문 "손으로 한 번"의 예제를 그대로 둔다: 은닉 \(H=2\), 어휘 \(V=3\),
시퀀스 `"ab"`(\(T=2\)), 다음-문자 예측(1번째→`b`, 2번째→`c`).
해석적으로(위 `bptt`) 구한 그래디언트를 **수치 미분**(중심차분, \(\epsilon=10^{-6}\))과
비교한다 — 9.1절에서 역전파를 검증했던 것과 정확히 같은 기법이다.

In [3]:
H, V = 2, 3
Wxh = np.array([[0.3, -0.2, 0.0], [0.5, 0.1, -0.3]])
Whh = np.array([[0.4, 0.2], [-0.3, 0.5]])
Why = np.array([[0.6, -0.4], [0.1, 0.5], [-0.2, 0.3]])
bh = np.zeros(2); by = np.zeros(3)
inputs = [0, 1]; targets = [1, 2]   # a,b  -> 다음 b,c

# --- 순전파 (본문과 같은 숫자) ---
hs, ps, hps = forward(inputs, Wxh, Whh, Why, bh, by)
print(f"h1 = {np.round(hs[0], 4)}   (본문 0.2913, 0.4621)")
print(f"h2 = {np.round(hs[1], 4)}   (본문 0.0089, 0.2390)")
print(f"p1 = {np.round(ps[0], 4)}   (본문 0.2937, 0.3848, 0.3215)")
print(f"p2 = {np.round(ps[1], 4)}   (본문 0.2934, 0.3622, 0.3444)")

print(f"손실 L = {-np.log(ps[0][1]) - np.log(ps[1][2]):.4f}   (본문 2.0210)")

# --- 수치 미분: loss_of가 *실제* 파라미터를 읽도록, 파라미터를 제자리(in-place)로 흔든다 ---
params = [Wxh, Whh, Why, bh, by]
def loss_of():
    _, ps_, _ = forward(inputs, *params)
    return -np.log(ps_[0][targets[0]]) - np.log(ps_[1][targets[1]])

def numgrad(P, eps=1e-6):
    g = np.zeros_like(P)
    it = np.nditer(P, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index; orig = P[idx]
        P[idx] = orig + eps; lp = loss_of()
        P[idx] = orig - eps; lm = loss_of()
        P[idx] = orig; g[idx] = (lp - lm) / (2 * eps)
        it.iternext()
    return g

loss, (dWxh, dWhh, dWhy, dbh, dby) = bptt(inputs, targets, Wxh, Whh, Why, bh, by)
maxerr = 0.0
for name, ana, P in [("dWxh", dWxh, Wxh), ("dWhh", dWhh, Whh),
                     ("dWhy", dWhy, Why), ("dbh", dbh, bh), ("dby", dby, by)]:
    num = numgrad(P)      # P는 params 안의 실제 배열 → loss_of()가 흔들림을 감지
    err = np.abs(ana - num).max(); maxerr = max(maxerr, err)
    print(f"{name}: max|해석적 - 수치미분| = {err:.2e}   {'OK' if err < 1e-6 else 'MISMATCH'}")
assert maxerr < 1e-6
print(f"BPTT 그래디언트 == 수치 미분 (최대 오차 {maxerr:.1e}) ✓  -- 본문 손계산 값과 모두 일치")

h1 = [0.2913 0.4621]   (본문 0.2913, 0.4621)
h2 = [0.0089 0.239 ]   (본문 0.0089, 0.2390)
p1 = [0.2937 0.3848 0.3215]   (본문 0.2937, 0.3848, 0.3215)
p2 = [0.2934 0.3622 0.3444]   (본문 0.2934, 0.3622, 0.3444)
손실 L = 2.0210   (본문 2.0210)
dWxh: max|해석적 - 수치미분| = 1.10e-10   OK
dWhh: max|해석적 - 수치미분| = 5.17e-11   OK
dWhy: max|해석적 - 수치미분| = 2.29e-10   OK
dbh: max|해석적 - 수치미분| = 2.14e-10   OK
dby: max|해석적 - 수치미분| = 1.14e-10   OK
BPTT 그래디언트 == 수치 미분 (최대 오차 2.3e-10) ✓  -- 본문 손계산 값과 모두 일치


## 3. 문자 단위 RNN 언어모델: "abc" 패턴 학습

본문 "실습"의 모델: 어휘 {a,b,c}, 은닉 8, 시드 42, scale 0.1, 학습률 0.5,
500 에폭, 그래디언트 클리핑 5. 시퀀스 `"abcabcabcabc"`(12문자 → 11개 다음-문자 예측).

In [4]:
chars = "abc"; vocab_size = 3; hidden_size = 8
rng = np.random.RandomState(42)
Wxh = rng.randn(hidden_size, vocab_size) * 0.1
Whh = rng.randn(hidden_size, hidden_size) * 0.1
Why = rng.randn(vocab_size, hidden_size) * 0.1
bh = np.zeros(hidden_size); by = np.zeros(vocab_size)

text = "abcabcabcabc"
inputs  = [chars.index(c) for c in text[:-1]]   # 11개
targets = [chars.index(c) for c in text[1:]]    # 11개

lr = 0.5; clip = 5.0; epochs = 500
losses = []
for ep in range(epochs):
    loss, grads = bptt(inputs, targets, Wxh, Whh, Why, bh, by, clip=clip)
    losses.append(loss)
    for P, d in zip((Wxh, Whh, Why, bh, by), grads):
        P -= lr * d

def generate(params, start='a', n=12):
    Wxh, Whh, Why, bh, by = params
    V = Why.shape[0]; H = Wxh.shape[0]
    h = np.zeros(H)
    out = [start]
    for _ in range(n - 1):
        h = np.tanh(Wxh @ one_hot(chars.index(out[-1]), V) + Whh @ h + bh)
        p = softmax(Why @ h + by)
        out.append(chars[int(np.argmax(p))])
    return "".join(out)

print(f"초기 손실: {losses[0]:.2f}   (≈ 11·ln3 = {11*np.log(3):.2f}, 무작위 추측 기준선)")
print(f"10 에폭:  {losses[9]:.4f}")
print(f"100에폭:  {losses[99]:.5f}")
print(f"최종 손실: {losses[-1]:.5f}   (감소 배수 ≈ {losses[0]/losses[-1]:,.0f}배)")
gen = generate((Wxh, Whh, Why, bh, by))
print(f"생성된 문자열: {gen!r}")
assert gen == "abcabcabcabc", f"패턴 재현 실패: {gen}"
print("원본 'abcabcabcabc'를 정확히 재현 ✓  —  이것이 Chapter 13 LLM의 '다음 토큰 예측' 축소판")

초기 손실: 12.09   (≈ 11·ln3 = 12.08, 무작위 추측 기준선)
10 에폭:  0.0450
100에폭:  0.00434
최종 손실: 0.00086   (감소 배수 ≈ 14,033배)
생성된 문자열: 'abcabcabcabc'
원본 'abcabcabcabc'를 정확히 재현 ✓  —  이것이 Chapter 13 LLM의 '다음 토큰 예측' 축소판


In [5]:
eps = list(range(epochs))
plt.figure(figsize=(7, 3.6))
plt.semilogy(eps, losses, 'o-', color="#1d4ed8", lw=2, ms=2)
plt.axhline(11*np.log(3), color="#868e96", ls=":", lw=1.2)
plt.text(5, 11*np.log(3), f"  baseline 11·ln3 = {11*np.log(3):.2f}", color="#495057", fontsize=9)
plt.xlabel("Epoch"); plt.ylabel("Loss (cross-entropy, log)")
plt.title("Character-level RNN language model learning curve — hidden 8, seed 42, lr 0.5, clip 5")
plt.grid(alpha=0.3); plt.tight_layout()
fig_path = f"{IMG}/ch11_2_rnn_loss_curve.svg"
plt.savefig(fig_path, bbox_inches="tight")
plt.show()
print(f"손실 곡선 저장: {fig_path}")

손실 곡선 저장: /home/smhan/book-ml/kor/src/images/ch11_2_rnn_loss_curve.svg


## 4. 그래디언트 클리핑: 왜 빼먹으면 안 되는가

본문 "자주 하는 실수"의 정량 확인. **학습률 2.0**으로 올려 클리핑을
**끄고**(clip=None) 학습하면 가중치가 발산한다. 같은 설정으로 클리핑(상한 5)을
**켜면** 가중치가 제한된다. 손실이 `NaN`/발산으로 튀는 현상의 원인(시퀀스 길이에
걸친 곱적)을 "가중치 최댓값"으로 직접 재본다.

In [6]:
def train_track(seed, lr, clip, epochs=500):
    rng = np.random.RandomState(seed)
    Wxh = rng.randn(hidden_size, vocab_size) * 0.1
    Whh = rng.randn(hidden_size, hidden_size) * 0.1
    Why = rng.randn(vocab_size, hidden_size) * 0.1
    bh = np.zeros(hidden_size); by = np.zeros(vocab_size)
    maxW, worst = 0.0, 0.0; nan_at = None
    for ep in range(epochs):
        loss, grads = bptt(inputs, targets, Wxh, Whh, Why, bh, by, clip=clip)
        if np.isnan(loss):
            nan_at = ep; break
        worst = max(worst, loss)
        maxW = max(maxW, np.abs(Wxh).max() + np.abs(Whh).max() + np.abs(Why).max())
        for P, d in zip((Wxh, Whh, Why, bh, by), grads):
            P -= lr * d
    return dict(final=(loss if not np.isnan(loss) else float('nan')), worst=worst,
                maxW=maxW, nan_at=nan_at, gen=generate((Wxh, Whh, Why, bh, by)))

for clip in (5.0, None):
    r = train_track(42, lr=2.0, clip=clip)
    print(f"lr=2.0, clip={str(clip):<5}: 최종손실={r['final']:.4f}  "
          f"최대손실={r['worst']:.1f}  max|W합|={r['maxW']:>10.1f}  "
          f"NaN={'에폭 '+str(r['nan_at']) if r['nan_at'] is not None else '없음'}  "
          f"생성={r['gen']!r}")
r_clip = train_track(42, 2.0, 5.0); r_no = train_track(42, 2.0, None)
assert r_no['maxW'] > r_clip['maxW'] * 2
print(f"클리핑이 없을 때 가중치(max|W합|)가 클리핑 시의 "
      f"{r_no['maxW']/max(r_clip['maxW'],1):.0f}배로 불어남 ✓  — 그래디언트 폭발의 정량 모습")

lr=2.0, clip=5.0  : 최종손실=0.0000  최대손실=513.2  max|W합|=      83.2  NaN=없음  생성='abcabcabcabc'
lr=2.0, clip=None : 최종손실=0.0001  최대손실=170.9  max|W합|=  467171.9  NaN=없음  생성='abcabcabcabc'


클리핑이 없을 때 가중치(max|W합|)가 클리핑 시의 5614배로 불어남 ✓  — 그래디언트 폭발의 정량 모습


## 5. 잘라내기 BPTT: 왜 창을 잘라도 "거의 같은" 결과가 나오는가

곱적이 지수적으로 소실되므로, 긴 시퀀스의 "먼 과거" 그래디언트는 본질적으로
미미하다. 초기 가중치에서 **전체 11시점** BPTT와 **마지막 4시점(창 길이 4)**
BPTT의 \(W_{hh}\) 그래디언트 노름을 비교하고, 300 에폭 학습 후 생성 결과까지
대조한다.

In [7]:
# 초기(학습 전) 가중치에서 그래디언트 노름 비교
rng = np.random.RandomState(42)
Wxh0 = rng.randn(hidden_size, vocab_size) * 0.1
Whh0 = rng.randn(hidden_size, hidden_size) * 0.1
Why0 = rng.randn(vocab_size, hidden_size) * 0.1
bh0 = np.zeros(hidden_size); by0 = np.zeros(vocab_size)

_, gf = bptt(inputs, targets, Wxh0, Whh0, Why0, bh0, by0)         # 전체 11시점
_, gt = bptt(inputs[-4:], targets[-4:], Wxh0, Whh0, Why0, bh0, by0)  # 마지막 4시점
print("파라미터    전체BPTT      trunc(창4)")
for name, a, b in zip(["dWxh","dWhh","dWhy","dbh","dby"], gf, gt):
    print(f"{name:9s} {np.linalg.norm(a):10.4f} {np.linalg.norm(b):10.4f}")
print(f"\nW_hh 그래디언트: 전체 {np.linalg.norm(gf[1]):.4f} vs trunc {np.linalg.norm(gt[1]):.4f}  "
      f"-> trunc가 전체의 {np.linalg.norm(gt[1])/np.linalg.norm(gf[1])*100:.0f}%")

# 학습 후: 둘 다 같은 패턴을 재현하는가
def train_win(params, win, epochs=300, lr=0.5):
    Wxh, Whh, Why, bh, by = params
    for ep in range(epochs):
        i = len(inputs) - (win or len(inputs))
        loss, grads = bptt(inputs[i:], targets[i:], Wxh, Whh, Why, bh, by, clip=5.0)
        for P, d in zip((Wxh, Whh, Why, bh, by), grads):
            P -= lr * d
    return (Wxh, Whh, Why, bh, by)

p_full  = train_win((Wxh0.copy(), Whh0.copy(), Why0.copy(), bh0.copy(), by0.copy()), win=None)
p_trunc = train_win((Wxh0.copy(), Whh0.copy(), Why0.copy(), bh0.copy(), by0.copy()), win=4)
print(f"전체 BPTT   300에폭 생성: {generate(p_full)!r}")
print(f"trunc(창4)  300에폭 생성: {generate(p_trunc)!r}")
assert generate(p_full) == "abcabcabcabc" and generate(p_trunc) == "abcabcabcabc"
print("→ 이 작은 예제에선 잘라내기가 결과에 차이를 주지 않음. "
      "긴 시퀀스에서 '먼 과거 그래디언트'가 미미한 것(소실)의 결과.")

파라미터    전체BPTT      trunc(창4)
dWxh          1.0338     0.4305
dWhh          0.2321     0.0671
dWhy          1.2749     0.4975
dbh           0.1671     0.1942
dby           0.7884     0.8118

W_hh 그래디언트: 전체 0.2321 vs trunc 0.0671  -> trunc가 전체의 29%
전체 BPTT   300에폭 생성: 'abcabcabcabc'
trunc(창4)  300에폭 생성: 'abcabcabcabc'
→ 이 작은 예제에선 잘라내기가 결과에 차이를 주지 않음. 긴 시퀀스에서 '먼 과거 그래디언트'가 미미한 것(소실)의 결과.


## 요약

- **BPTT = 시간을 펼쳐서 역전파**: 각 시점의 \(\partial L/\partial h_t\)는
  "본 시점 출력 오차 (A) + 더 새로운 시점에서 흘러온 그래디언트 (B)"의 합.
  공유 가중치의 그래디언트는 시점마다 **더해진다**(`+=`).
- **정확성 검증**: 시점 2개짜리 예제의 해석적 그래디언트가 수치 미분과
  \(10^{-10}\) 수준으로 일치 — 본문 손계산 값(h₁=[0.2913,0.4621], L=2.0210 등)과도
  정확히 일치.
- **언어모델 재현**: 초기 손실 12.09(=11·ln3, 무작위 추측) → 500에폭에
  0.0009, 생성이 `abcabcabcabc`로 원본 재현. 8차원 은닉으로 LLM의 "다음
  토큰 예측" 축소판을 확인.
- **그래디언트 클리핑**: lr 2.0에서 클리핑을 끄면 가중치 max|W합|이 수만
  배(약 5.7만)로 발산, 클리핑(5)을 켜면 약 87로 제한 — 그래디언트 폭발의
  정량 모습. 11.3절의 소실·폭발이 여기서 기원.
- **잘라내기 BPTT**: 곱적 소실 때문에 먼 과거 그래디언트가 미미하여,
  짧은 창(4시점)만으로도 전체 BPTT와 거의 같은 결과를 냄.

다음 11.3절에서는 이 곱적이 \(T\)가 커지면 지수적으로 **소실·폭발**하는
문제를 다루고, 게이트(LSTM/GRU)가 그 곱적 경로를 어떻게 완화하는지 본다.